# <font color="steelblue">10. Medidas de dispersión: ¿son los datos parecidos o muy distintos?</font>

*Unidad 2. Estadística descriptiva* · Sesión teórica 10 de 30

**Autoría**:

*   Fernando Borrás Rocher (f.borras@umh.es), autor de la idea original
*   Juan Francisco Monge Ivars
*   José Santiago García Cremades

Departamento de Estadística, Matemáticas e Informática. Universidad Miguel Hernández de Elche.

Basado en [SIMUSTAT](https://unidigitalsimustat.umh.es/) (proyecto UNIDIGITAL-UMH, CC BY-SA 4.0).

**Asignatura**: Estadística (1200), 1.º curso, Grado en Periodismo.

**Licencia**: <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="CC BY-SA 4.0" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a> Creative Commons Atribución-CompartirIgual 4.0.

> ⚠️ **Los datos de este curso son simulados.** Imitan la forma de fuentes reales (INE, CIS, audímetros, agencias de verificación), pero ninguna cifra es real. Úsalos para aprender; para publicar, ve siempre a la fuente oficial.

## <font color='steelblue'>Antes de empezar: configura el cuaderno</font>

Ejecuta **una vez** la celda de abajo (botón ▶). Instala la librería de las preguntas, descarga los datos (`audiencias_tv.csv`, `salarios_redaccion.csv`) y las preguntas de autoevaluación. Si todo va bien verás **`Cuaderno configurado`**.

> 💡 Si Colab se desconecta, los ficheros se borran: vuelve a ejecutar esta celda.

In [ ]:
#@title <font color="steelblue" size="+1"> Ejecutar configuración { display-mode: "form" }
import os, sys, json, base64, subprocess, urllib.request

RAW = "https://raw.githubusercontent.com/fborrasumh/statperiodismo/main/"
DATOS = ["audiencias_tv.csv", "salarios_redaccion.csv"]
PREGUNTAS = ["q_P10_1.json", "q_P10_2.json", "q_P10_3.json", "q_P10_4.json"]

try:
    from jupyterquiz import display_quiz
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jupyterquiz==2.9.6.4"], check=True)
    from jupyterquiz import display_quiz

for carpeta, ficheros in (("datos/", DATOS), ("autoeval/", PREGUNTAS)):
    for f in ficheros:
        if not os.path.exists(f):
            urllib.request.urlretrieve(RAW + carpeta + f, f)

def pregunta(fichero):
    """Muestra una pregunta interactiva de autoevaluación."""
    display_quiz(json.loads(base64.b64decode(open(fichero, "rb").read()).decode("utf8")))

print("Cuaderno configurado")

## <font color='steelblue'>Introducción</font>

Dos cadenas pueden tener el mismo share medio y comportamientos opuestos: una siempre igual, la otra con programas que arrasan y otros que no ve nadie. La media no lo dice; las **medidas de dispersión**, sí. Sin ellas, cualquier promedio es una media verdad.

**Qué necesitas saber antes**: los cuadernos 8 y 9.

## <font color='steelblue'>Qué vas a aprender</font>

Al terminar este cuaderno sabrás:

1. Calcular el rango y el rango intercuartílico.
2. Calcular e interpretar la varianza y la desviación típica.
3. Comparar la dispersión de variables con distinta escala con el coeficiente de variación.
4. Resumir una variable con media y desviación típica en una noticia.

## <font color='steelblue'>Contenidos</font>

1. Rango y rango intercuartílico
2. Varianza y desviación típica
3. Coeficiente de variación
4. Cómo contarlo

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 4)
pd.set_option('display.max_columns', 20)

## <font color='steelblue'>1. Rango y rango intercuartílico</font>

In [ ]:
audiencias = pd.read_csv("audiencias_tv.csv")
salarios = pd.read_csv("salarios_redaccion.csv")

- **Rango** = máximo − mínimo. Fácil, pero depende solo de los dos valores más extremos.
- **Rango intercuartílico** (RIC) = $Q_3 - Q_1$. Mide la amplitud de la mitad central y no le afectan los extremos.

In [ ]:
sh = audiencias["share"]
print("Rango:", round(sh.max() - sh.min(), 1))
print("RIC:  ", round(sh.quantile(0.75) - sh.quantile(0.25), 1))

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P10_1.json")

## <font color='steelblue'>2. Varianza y desviación típica</font>

La **varianza** mide cuánto se alejan, de media, los datos de su media (al cuadrado):

$$s^2 = \frac{\sum (x_i - \bar{x})^2}{n - 1}$$

Como está en unidades al cuadrado (puntos de share²), se usa su raíz, la **desviación típica** $s$, que está en las mismas unidades que los datos.

**Qué vamos a hacer**: calcularla paso a paso con cinco programas y comprobarla con Pandas.

In [ ]:
x = np.array([12.0, 9.0, 15.0, 10.0, 14.0])
media = x.mean()
desvios = x - media
print("Desvíos respecto a la media:", desvios)
varianza = (desvios ** 2).sum() / (len(x) - 1)
print("Varianza:", varianza, "  Desviación típica:", round(np.sqrt(varianza), 2))
print("Con Pandas:", round(pd.Series(x).std(), 2))

**Qué ha pasado**: los desvíos suman cero (siempre ocurre), por eso se elevan al cuadrado. La desviación típica de 2,5 puntos indica que estos programas se separan, en promedio, unos 2,5 puntos del share medio (12).

Se divide entre $n-1$ porque trabajamos con una **muestra**; es lo que hace Pandas por defecto.

In [ ]:
por_cadena = audiencias.groupby("cadena")["share"].agg(["mean", "std"]).round(1)
por_cadena

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escribe el número y pulsa Enter:

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P10_2.json")

## <font color='steelblue'>3. Coeficiente de variación</font>

Para comparar la dispersión de variables con **unidades o tamaños distintos** (share en % y espectadores en miles, o salarios de becarios y directivos) se usa el **coeficiente de variación**:

$$CV = \frac{s}{\bar{x}} \times 100$$

No tiene unidades. Como orientación, un CV por debajo del 30 % indica datos bastante homogéneos.

In [ ]:
def cv(serie):
    return round(serie.std() / serie.mean() * 100, 1)

print("CV del share:       ", cv(audiencias["share"]), "%")
print("CV de espectadores: ", cv(audiencias["espectadores_miles"]), "%")
salarios.groupby("puesto")["salario_bruto_anual"].apply(cv)

**Cómo leerlo**: los espectadores varían mucho más que el share, porque dependen también de la franja (a medianoche hay menos gente frente al televisor). Y dentro de cada puesto los salarios son bastante homogéneos.

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P10_3.json")

## <font color='steelblue'>4. Cómo contarlo</font>

En una noticia la dispersión se cuenta mejor con **ejemplos y rangos** que con la palabra «desviación típica»:

- «La mitad de los programas tiene un share de entre 6 y 12 puntos» (cuartiles).
- «Los sueldos de la redacción van de menos de 15.000 a casi 120.000 euros» (rango).
- «Los informativos de la Cadena 3 son los más regulares: casi todos rondan el 13 %» (desviación típica pequeña).

En los gráficos, las barras de error o las cajas transmiten la dispersión sin fórmulas.

In [ ]:
q1, q3 = audiencias["share"].quantile([0.25, 0.75])
print(f"La mitad de los programas tiene un share de entre {q1:.0f} y {q3:.0f} puntos.")

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P10_4.json")

## <font color='steelblue'>Reto aprendiz</font>

Resuelve estos ejercicios en la celda vacía de debajo de cada enunciado. Si te atascas, vuelve a la sección indicada. La solución está plegada: **inténtalo antes de mirarla**.

### **Ejercicio 1** *(sección 1)*

Calcula el rango y el RIC del salario. ¿Cuál describe mejor la variabilidad de la mayoría de la plantilla?

<details><summary>Ver solución</summary>

```python
s = salarios["salario_bruto_anual"]
print(s.max() - s.min(), s.quantile(0.75) - s.quantile(0.25))
```

El RIC: el rango lo marcan los sueldos de dirección.
</details>

### **Ejercicio 2** *(sección 2)*

¿Qué género de programa tiene el share más regular (menor desviación típica)?

<details><summary>Ver solución</summary>

```python
audiencias.groupby("genero")["share"].std().sort_values()
```
</details>

### **Ejercicio 3** *(sección 3)*

Compara el CV de los salarios de autónomos y de indefinidos.

<details><summary>Ver solución</summary>

```python
salarios.groupby("contrato")["salario_bruto_anual"].apply(lambda x: x.std() / x.mean() * 100)
```
</details>

### **Ejercicio 4** *(sección 4)*

Escribe con una f-string una frase del tipo «La mitad de la plantilla cobra entre X y Y euros».

<details><summary>Ver solución</summary>

```python
q1, q3 = salarios["salario_bruto_anual"].quantile([0.25, 0.75])
print(f"La mitad de la plantilla cobra entre {q1:,.0f} y {q3:,.0f} euros")
```
</details>

## <font color='steelblue'>Resumen: ideas clave</font>

- El **rango** depende de los extremos; el **RIC** mide la mitad central y es robusto.
- La **desviación típica** mide la distancia típica de los datos a la media, en sus mismas unidades.
- El **coeficiente de variación** compara dispersiones de variables con escalas distintas.
- En prensa, cuenta la dispersión con **rangos y ejemplos**: «la mitad está entre X e Y».

## <font color='steelblue'>Glosario</font>

| Término | Significado |
| :-- | :-- |
| **Rango** | Diferencia entre el máximo y el mínimo. |
| **Varianza** | Media de los desvíos al cuadrado (dividiendo por n − 1). |
| **Desviación típica** | Raíz cuadrada de la varianza. |
| **Coeficiente de variación** | Desviación típica dividida por la media, en %. |

## <font color='steelblue'>Errores frecuentes y cómo resolverlos</font>

| Síntoma | Causa probable | Solución |
| :-- | :-- | :-- |
| La desviación típica de NumPy no coincide con la de Pandas | NumPy divide entre n y Pandas entre n − 1 | Usa `np.std(x, ddof=1)` |
| `NameError: name 'pd' is not defined` | No se ejecutó la celda de importación | Ejecuta las celdas desde el principio, en orden |
| `FileNotFoundError` al leer un fichero | No se ejecutó la configuración o Colab se reinició | Ejecuta la celda *Configura el cuaderno* |
| La pregunta de autoevaluación no aparece | Falta la configuración | Ejecuta la celda de configuración y luego la pregunta |

## <font color='steelblue'>Para seguir</font>

En el cuaderno 11 verás cómo detectar valores atípicos y describir la forma de una distribución.

**Siguiente cuaderno**: [11. Valores atípicos, puntuaciones z y forma de la distribución](https://colab.research.google.com/github/fborrasumh/statperiodismo/blob/main/cuadernos/P11_atipicos_y_forma.ipynb)